## Create an example database schema and load it with data

In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    String,
    Integer,
    select,
    column,
)

In [3]:
engine = create_engine("sqlite:///:memory:", future=True)
metadata_obj = MetaData()

Create database tables based on the table data from the BMO pdf.

In [4]:
# create city SQL table
table_name = "summary_of_the_methods_to_identify_persons_and_associated_record_keeping_obligations"

methods_to_identify_persons = Table(
    table_name,
    metadata_obj,
    Column("identification_method", String(30), primary_key=True),
    Column("documents_or_information_to_review", String(500)),
    Column("identification_details_that_must_match", String(100), nullable=False),
    Column("information_that_must_be_recorded", String(500), nullable=False),
)

metadata_obj.create_all(engine)

Manually populate the database from the table in Annex 1. Please only run this one time, or it will fail when attempting to insert duplicate data.

In [5]:
from sqlalchemy import insert

rows = [
    {
        "identification_method": "government-issued photo identification", 
        "documents_or_information_to_review": "photo identification document issued by a government (not a municipal government) that is authentic, valid and current", 
        "identification_details_that_must_match": "name and photograph",
        "information_that_must_be_recorded": "person's name, date of verification, type of document, document number, province or state and country issued the document, expiry date (if applicable)"
    },
    {
        "identification_method": "credit file", 
        "documents_or_information_to_review": "valid and curent information from a canadian credit file that has been in existance for at least three years where information is derived more than one source", 
        "identification_details_that_must_match": "name, address, and date of birth",
        "information_that_must_be_recorded": "person's name, date you consulted/searched the credit file, name of the credit bureau or third party vendor, persons credit file number"
    },
    {
        "identification_method": "dual-process", 
        "documents_or_information_to_review": "valid and current information from two different reliable sources where neither the RE nor the person is a source", 
        "identification_details_that_must_match": "a combination of the following: name and address, name and date of birth, or name and confirmation of a financial account",
        "information_that_must_be_recorded": "person's name, date you verified the information, name of two different sources used to verify the identity of the person, type of information referred to, account number or number associated with the information if account number exists"
    },
    {
        "identification_method": "affiliate or member", 
        "documents_or_information_to_review": "information in the records of the affiliate or the member for the method used", 
        "identification_details_that_must_match": "name, address and date of birth",
        "information_that_must_be_recorded": "person's name, date you verified the identity of the person, name of affiliate or member that previously verified the identity of the person, method used by the affiliate or member to verify the person's identity, information that the affiliate or member recorded based on the method used"
    },
    {
        "identification_method": "reliance", 
        "documents_or_information_to_review": "be satisfied that the information from the other RE or affiliated foreign entity is valid and current and that the person's identity was verified by using the government-issued photo identification, credit file or dual-process methods, or where the identity was verified prior to June 1, 2021, that the person's identity was verified using one of the methods in force in the PCMLTFR at that time ", 
        "identification_details_that_must_match": "The identification details listed under the identification method used",
        "information_that_must_be_recorded": "person's name, the written agreement or arrangement with the other RE or affiliated foreign entity is valid and current and that the person's identity was verified by using the government- issued photo identification, credit file or dual-process methods or Where the identity was verified prior to June 1, 2021, that the person's identity was verified using one of the methods in force in the PCMLTFR at that time entity for the purpose of verifying a person's identity"
    }
]
for row in rows:
    stmt = insert(methods_to_identify_persons).values(**row)
    with engine.begin() as connection:
        cursor = connection.execute(stmt)

Optional: print your tables and see the data inside

In [6]:
with engine.connect() as connection:
    cursor = connection.exec_driver_sql("SELECT * FROM summary_of_the_methods_to_identify_persons_and_associated_record_keeping_obligations")
    print(cursor.fetchall())

[('government-issued photo identification', 'photo identification document issued by a government (not a municipal government) that is authentic, valid and current', 'name and photograph', "person's name, date of verification, type of document, document number, province or state and country issued the document, expiry date (if applicable)"), ('credit file', 'valid and curent information from a canadian credit file that has been in existance for at least three years where information is derived more than one source', 'name, address, and date of birth', "person's name, date you consulted/searched the credit file, name of the credit bureau or third party vendor, persons credit file number"), ('dual-process', 'valid and current information from two different reliable sources where neither the RE nor the person is a source', 'a combination of the following: name and address, name and date of birth, or name and confirmation of a financial account', "person's name, date you verified the infor

### Initialize Ollama

In [7]:
%pip install langchain-community langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_ollama.chat_models import ChatOllama
llm = ChatOllama(model="llama3.2:latest", temperature=0.1)

## Create Agents and Tools

In [9]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


### Create SQL tool that connects to your database

In [10]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [11]:
db = SQLDatabase(engine)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

### Create Agent to manage that SQL tool

In [12]:
from langchain import hub
from langgraph.prebuilt import create_react_agent

In [13]:
# Pull default SQL agent prompt
prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")
system_message = prompt_template.format(dialect="SQLite", top_k=5)

# Create agent
agent_executor = create_react_agent(
    model=llm, tools=tools, state_modifier=system_message, debug=False
)

/Users/egarcia/miniconda3/envs/basic-agent/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


## Execute your queries

In [14]:
from langchain_core.messages import HumanMessage

In [21]:
messages = {
    "messages": [HumanMessage(content="What ways can be used to prove identity for record keeping purposes? There may be relevant data in my SQL database to assist you with this question. If you find something, please incorperate it into your response.")],
    }

In [22]:
result = agent_executor.invoke(messages)
print(f"Final Result: {result['messages'][-1].content} \n")

print("Step by step execution")
for message in result['messages']:
   print(message.pretty_repr())

Final Result: Based on the schema of the `user` table, I found that there is a column named `identification_number` which can be used to prove identity. Additionally, there is a column named `recordKeepingPurpose` with values 0 or 1, where 1 indicates that the user's identity needs to be proven for record keeping purposes.

Here are some ways to prove identity for record keeping purposes:

1. **Government-issued ID**: A valid government-issued ID such as a driver's license, passport, or state ID can be used to prove identity.
2. **Social Security Number**: In the United States, a Social Security Number (SSN) is a unique identifier assigned to U.S. citizens and certain non-citizens. It can be used to prove identity for record keeping purposes.
3. **Biometric Data**: Biometric data such as fingerprints, facial recognition, or iris scans can be used to verify an individual's identity.
4. **Digital Certificates**: Digital certificates issued by a trusted authority can be used to prove iden